In [4]:
# Sentiment analysis using lemmatization
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Download datasets required for Lemmatization and POS tagging
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Initialize the Lemmatizer
lemmatizer = WordNetLemmatizer()

# --- 1. HELPER TO CONVERT NLTK TAGS TO WORDNET TAGS ---
def get_wordnet_pos(nltk_tag):
    """
    Lemmatization works best when given a Part-of-Speech (POS) tag.
    This helper maps default NLTK POS tags to WordNet-compatible tags.
    """
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default fallback

# --- 2. DEFINE THE LEMMATIZATION CLEANING FUNCTION ---
def lemmatize_text(text):
    """
    Tokenizes, POS tags, and lemmatizes a string of text.
    """
    # Tokenize text into individual words
    tokens = nltk.word_tokenize(text.lower())

    # Get POS tags for the words (e.g., [('running', 'VBG')])
    pos_tags = nltk.pos_tag(tokens)

    # Lemmatize each word using its context-driven POS tag
    clean_tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]

    # Recombine back into a single clean string
    return " ".join(clean_tokens)


# --- 3. PREPARE THE TRAINING DATA ---
# Let's build a simple sentiment classifier (Positive vs Negative)
corpus = [
    "The actors were performing beautifully and dancing wonderfully!",
    "The cameras stopped working and the screens blurred out.",
    "I thoroughly enjoyed the pieces; they are hidden gems.",
    "He hates waiting in queues, everything moves so slowly.",
    "She loves how smooth the application runs on older devices."
]
labels = ["positive", "negative", "positive", "negative", "positive"]


# --- 4. APPLY LEMMATIZATION ---
print("--- Text Transformation Example ---")
for raw_text in corpus:
    print(f"Raw:   {raw_text}")
    print(f"Clean: {lemmatize_text(raw_text)}\n")

cleaned_corpus = [lemmatize_text(text) for text in corpus]


# --- 5. TRAIN THE MACHINE LEARNING MODEL ---
# Using a pipeline that vectorizes text and trains a classifier
ml_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression())
])

# Train the pipeline on the normalized lemmas
ml_pipeline.fit(cleaned_corpus, labels)


# --- 6. PREDICT ON NEW UNSEEN TEXT ---
new_reviews = [
    "The camera stops working easily.",  # Contains 'stops' instead of 'stopped'
    "They love the performances!"        # Contains 'love' and 'performances'
]

# Clean incoming data using the exact same normalization logic
cleaned_new_reviews = [lemmatize_text(text) for text in new_reviews]
print(cleaned_new_reviews)

# Predict classes
predictions = ml_pipeline.predict(cleaned_new_reviews)

print("--- Final Predictions ---")
for doc, pred in zip(new_reviews, predictions):
    print(f"Document: '{doc}' -> Predicted Sentiment: {pred.upper()}")

--- Text Transformation Example ---
Raw:   The actors were performing beautifully and dancing wonderfully!
Clean: the actor be perform beautifully and dance wonderfully !

Raw:   The cameras stopped working and the screens blurred out.
Clean: the camera stop work and the screen blur out .

Raw:   I thoroughly enjoyed the pieces; they are hidden gems.
Clean: i thoroughly enjoy the piece ; they be hidden gem .

Raw:   He hates waiting in queues, everything moves so slowly.
Clean: he hat wait in queue , everything move so slowly .

Raw:   She loves how smooth the application runs on older devices.
Clean: she love how smooth the application run on old device .

['the camera stop work easily .', 'they love the performance !']
--- Final Predictions ---
Document: 'The camera stops working easily.' -> Predicted Sentiment: POSITIVE
Document: 'They love the performances!' -> Predicted Sentiment: POSITIVE


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [19]:
# Note that above model is not predicting "The camera stops working easily." as negative
# This is due to
#  1) We have less training data
#  2) By default TFIDF vectorizer uses unigrams
# So we can add more training data first and see whether its predicting correctly or not

# --- 3. PREPARE THE TRAINING DATA ---
# Let's build a simple sentiment classifier (Positive vs Negative)
corpus = [
    "The actors were performing beautifully and dancing wonderfully!",
    "The cameras stopped working and the screens blurred out.",
    "I thoroughly enjoyed the pieces; they are hidden gems.",
    "He hates waiting in queues, everything moves so slowly.",
    "She loves how smooth the application runs on older devices.",
    "Today camera stopped working",
    "Water stopped running suddently",
    "I love stage performances"
]
labels = ["positive", "negative", "positive", "negative", "positive", "negative", "negative", "positive"]

# --- 4. APPLY LEMMATIZATION ---
print("--- Text Transformation Example ---")
for raw_text in corpus:
    print(f"Raw:   {raw_text}")
    print(f"Clean: {lemmatize_text(raw_text)}\n")

cleaned_corpus = [lemmatize_text(text) for text in corpus]


# --- 5. TRAIN THE MACHINE LEARNING MODEL ---
# Using a pipeline that vectorizes text and trains a classifier
# Added ngram_range=(1, 2) to capture word pairs like "stop work"
ml_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('classifier', LogisticRegression())
])

# Train the pipeline on the normalized lemmas
ml_pipeline.fit(cleaned_corpus, labels)


# --- 6. PREDICT ON NEW UNSEEN TEXT ---
new_reviews = [
    "The camera stops working easily.",  # Contains 'stops' instead of 'stopped'
    "They love the performances!",        # Contains 'love' and 'performances'
]

# Clean incoming data using the exact same normalization logic
cleaned_new_reviews = [lemmatize_text(text) for text in new_reviews]
print(cleaned_new_reviews)

# Predict classes
predictions = ml_pipeline.predict(cleaned_new_reviews)

print("--- Final Predictions ---")
for doc, pred in zip(new_reviews, predictions):
    print(f"Document: '{doc}' -> Predicted Sentiment: {pred.upper()}")

--- Text Transformation Example ---
Raw:   The actors were performing beautifully and dancing wonderfully!
Clean: the actor be perform beautifully and dance wonderfully !

Raw:   The cameras stopped working and the screens blurred out.
Clean: the camera stop work and the screen blur out .

Raw:   I thoroughly enjoyed the pieces; they are hidden gems.
Clean: i thoroughly enjoy the piece ; they be hidden gem .

Raw:   He hates waiting in queues, everything moves so slowly.
Clean: he hat wait in queue , everything move so slowly .

Raw:   She loves how smooth the application runs on older devices.
Clean: she love how smooth the application run on old device .

Raw:   Today camera stopped working
Clean: today camera stop work

Raw:   Water stopped running suddently
Clean: water stop run suddently

Raw:   I love stage performances
Clean: i love stage performance

['the camera stop work easily .', 'they love the performance !']
--- Final Predictions ---
Document: 'The camera stops working ea